# CFPB - Customer Complaint Analysis

The **Consumer Financial Protection Bureau** (CFPB) is a U.S. government regulatory agency established to protect consumers in the financial marketplace. The agency operates a centralized system where consumers can submit complaints regarding financial institutions, prompting the company to review and respond. 
According to the official 2025 statistics: *the CFPB received around $6.63M$ consumer complaints via its registered portal, toll-free number, postal mail, referrals from other regulatory bodies, etc., and routed around $5.98M$ complaints to over $4K$ companies for review with an expected response TAT of ${<}15days$.*\
These archived reports contain detailed EDA breakdown: [[annual-report_2025](https://www.consumerfinance.gov/data-research/research-reports/2025-consumer-response-annual-report/)].[[annual-report_2024](https://www.consumerfinance.gov/data-research/research-reports/2024-consumer-response-annual-report/)].[[annual-report_2023](https://www.consumerfinance.gov/data-research/research-reports/consumer-response-annual-report-2023/)].[[annual-report_2022](https://www.consumerfinance.gov/data-research/research-reports/2022-consumer-response-annual-report/)].[[annual-report_2021](https://www.consumerfinance.gov/data-research/research-reports/2021-consumer-response-annual-report/)]\
The portal states: <span style="font-family:'Courier New', monospace; font-style:italic;">"Complaint narratives are consumers’ descriptions of their experiences in their own words"</span>; the CFPB only performs PII masking before uploading the narratives to the database. Consider the CFPB's description of the data collection process: <span style="font-family:'Courier New', monospace; font-style:italic;">"The Consumer Complaint Database shows the consumer's original products, sub-products, issues, and sub-issues selections consistent with the options available on the form at the time the consumer submitted the complaint"</span>. i.e., the data annotation is done by individual customers themselves, not something directly actionable for CFPB staff.
<div style="border: 2px solid black; padding: 15px; font-size: 1.2em; font-weight: bold; border-radius: 5px; width: max-content;">
    It is more sensible to spend effort understanding the themes and patterns within tagged complaints than trying to automatically assign those tags.
</div>



# Contents

- [Extracting only the relevant data](#extracting-only-the-relevant-data)
- [Let's count](#lets-count)


## Extracting only the relevant data
As of 21-June-2026, the CFPB [Consumer Complaint Database](https://www.consumerfinance.gov/data-research/consumer-complaints/search/?dateRange=All&date_received_max=2026-06-20&date_received_min=2011-12-01&page=1&searchField=all&size=25&sort=created_date_desc&tab=List) hosted around $16M$ records dating back to 2011. Instead of downloading the entire database dump, we pulled a 3-month window of data via the CFPB API after accounting for the portal's 15-day publishing lag. Only records with written narratives were retrieved, thereby excluding at the server level the $98.1\%$ of records within that timeframe that contained only category tags and blank text.\
Although the database is a collection of self-reported experiences rather than a statistical sample of marketplace consumers, companies still use this complaint data to identify potential weaknesses in specific products or services and to make future improvements. 

In [60]:
Source_file = './data.csv'

import pandas as pd

Head = pd.read_csv(Source_file,
    header=0, index_col=False, dtype='string',
    nrows=5
).set_axis(range(1, 6),axis=0)
Head

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
1,2026-03-04T15:21:40.000Z,Debt collection,Credit card debt,Communication tactics,Frequent or repeated calls,I am submitting this complaint regarding Midla...,<NA>,ENCORE CAPITAL GROUP INC.,FL,33417,<NA>,Web,2026-04-09T17:24:57.000Z,Closed with explanation,Yes,19966418
2,2026-03-04T18:21:34.000Z,Debt collection,Medical debt,Attempts to collect debt not owed,Debt is not yours,Revco debt collectors called and had my correc...,<NA>,"Revco Management, LLC",NY,14226,<NA>,Web,2026-03-04T18:30:13.000Z,Closed with non-monetary relief,Yes,19972144
3,2026-03-04T21:57:47.000Z,Credit card,General-purpose credit card or charge card,Incorrect information on your report,Account information incorrect,CDR Genesis is reporting a collection account ...,<NA>,"Rowland Avenue Management, Inc. A/KA Columbia ...",TX,754XX,<NA>,Web,2026-04-10T04:43:28.000Z,Closed with explanation,Yes,19981404
4,2026-03-04T21:04:23.000Z,Checking or savings account,Savings account,Managing an account,Deposits and withdrawals,XXXX XXXX {$3800.00} taken out my savings and ...,Company believes it acted appropriately as aut...,NAVY FEDERAL CREDIT UNION,VA,23320,<NA>,Web,2026-04-10T15:50:00.000Z,Closed with explanation,Yes,19978982
5,2026-03-07T05:24:36.000Z,Checking or savings account,Checking account,Managing an account,Cashing a check,I ordered a checkbook for my checking account ...,<NA>,"HOPE BANCORP, INC.",CA,90015,<NA>,Web,2026-03-07T08:21:00.000Z,Closed with monetary relief,Yes,20063160


In [ ]:
Date_idx = pd.read_csv(Source_file,
    header=0, index_col=False,
    usecols=["Date received"], 
    parse_dates=["Date received"]
)["Date received"].dt.date # timestamp is system generated ISO8601, no need to clean it up. 
print(f'''#n_rows: {len(Date_idx)}\nmin_date: {Date_idx.min()}\nmax_date: {Date_idx.max()} ''')

#n_rows: 37620
min_date: 2026-03-04
max_date: 2026-06-01 


In [116]:
Data_Narrative = pd.read_csv(Source_file,
    header=0, index_col=False, dtype='string',
    usecols=["Product","Sub-product","Consumer complaint narrative"]
).rename(
    columns={"Sub-product":"SubProduct", "Consumer complaint narrative":"Narrative"}
).dropna() # Tag is not manually typed but drop-down options. Blank narratives are already filtered out at the server side. Very unlikely to have missing values; the step is only for safety measure.
for col in ["Product","SubProduct"]: 
    Data_Narrative[col] = Data_Narrative[col].str.lower().str.strip().str.replace(r'[^a-z0-9]+','_',regex=True).astype('category')    
Data_Narrative["Narrative"] = Data_Narrative["Narrative"].str.replace(r'\s+',' ',regex=True)
Data_Narrative.info()

<class 'pandas.DataFrame'>
RangeIndex: 37620 entries, 0 to 37619
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   Product     37620 non-null  category
 1   SubProduct  37620 non-null  category
 2   Narrative   37620 non-null  string  
dtypes: category(2), string(1)
memory usage: 48.5 MB


In [121]:
Summary = pd.pivot_table(Data_Narrative, 
    index="Product", 
    aggfunc={
        "SubProduct":'nunique',
        "Narrative":'count'
    },
    sort=False
).sort_values("Narrative",ascending=False)
Summary

,SubProduct,Narrative
Product,,
debt_collection,11,12658
checking_or_savings_account,4,6835
credit_card,2,6450
money_transfer_virtual_currency_or_money_service,7,2821
mortgage,8,2439
vehicle_loan_or_lease,2,1857
credit_reporting_or_other_personal_consumer_reports,2,1517
payday_loan_title_loan_personal_loan_or_advance_loan,8,1183
student_loan,2,1166


## Let's count 
Let us start with Tf-Idf to evaluate how important a keyword is to a specific narrative in relation to the whole complaint base.

In [105]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    max_df=0.8, min_df=10,
    max_features=100
)
Narrative_TfIdf = vectorizer.fit_transform(Data_Narrative["Narrative"])


In [103]:
vectorizer.get_feature_names_out()

array(['00', '15', 'access', 'account', 'accounts', 'act', 'address',
       'agreement', 'alleged', 'balance', 'bank', 'called', 'card',
       'charge', 'charges', 'check', 'claim', 'closed', 'collection',
       'company', 'complaint', 'complete', 'consumer', 'contacted',
       'credit', 'credit report', 'credit reporting', 'date', 'days',
       'debt', 'despite', 'did', 'dispute', 'documentation', 'email',
       'failed', 'failure', 'fair', 'fargo', 'federal', 'financial',
       'following', 'fraud', 'fraudulent', 'funds', 'immediately',
       'inaccurate', 'including', 'information', 'investigation', 'issue',
       'letter', 'loan', 'matter', 'money', 'multiple', 'notice',
       'number', 'original', 'paid', 'pay', 'payment', 'payments',
       'phone', 'practices', 'proof', 'provide', 'provided', 'received',
       'refund', 'regarding', 'report', 'reported', 'reporting',
       'request', 'requested', 'requesting', 'required', 'response',
       'review', 'sent', 'service